In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import emoji

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, accuracy_score, f1_score

In [ ]:
df = pd.read_csv("./data/train_with_features.csv")
test = pd.read_csv("./data/test_with_features.csv")

SEED = 42

X_train, X_validation, y_train, y_validation = train_test_split(df[['text', 'sentimiento', 'tiene_url']], df['target'], test_size=0.2, random_state=SEED)
mean_encoder = TargetEncoder(random_state=SEED)
one_hot_encoder = OneHotEncoder()
xgb_model = XGBClassifier(random_state=SEED)
sbert = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
def limpiar_texto(texto):
    # Reemplazar URLs por token
    texto = re.sub(r"http\S+|www\S+", " URL ", texto)
    # Reemplazar menciones (@usuario)
    texto = re.sub(r"@\w+", " USER ", texto)   
    # Reemplazar hashtags (dejando la palabra)
    texto = re.sub(r"#(\w+)", r" HASHTAG_\1 ", texto) 
    # Reemplazar emojis (usando su significado textual)
    texto = emoji.demojize(texto, language="en")  # ?? ? :cara_sonriente:
    texto = re.sub(r":([a-zA-Z0-9_]+):", r" EMOJI_\1 ", texto)
    # Eliminar caracteres especiales innecesarios
    texto = re.sub(r"[^a-zA-Z????????????0-9_\s]", " ", texto) 
    # Pasar a min?sculas
    texto = texto.lower() 
    # Quitar espacios m?ltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    
    return texto

In [ ]:
x_sentimiento = mean_encoder.fit_transform(X_train[['sentimiento']], y_train)
x_url = one_hot_encoder.fit_transform(X_train[['tiene_url']]).toarray()

clean_text =  X_train.text.map(limpiar_texto)
X_emb = sbert.encode(clean_text.tolist(), show_progress_bar=True)

X_encoded = np.hstack((X_emb, x_sentimiento, x_url))

parametros_xgb = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 5, 7, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'n_estimators': [50, 100, 200, 300],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'alpha': [0, 0.01, 0.1, 1, 10],
    'lambda': [0.01, 0.1, 1, 10],
    'objective': ['binary:logistic']
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=parametros_xgb,
    n_iter=15,
    scoring='f1',
    cv=3,
    random_state=SEED,
    n_jobs=-1
)

In [ ]:
random_search_xgb.fit(X_encoded, y_train)

In [ ]:
x_sentimiento_val = mean_encoder.transform(X_validation[['sentimiento']])
x_url_val = one_hot_encoder.transform(X_validation[['tiene_url']]).toarray()

clean_text_val =  X_validation.text.map(limpiar_texto)
x_emb_val = sbert.encode(clean_text_val.tolist(), show_progress_bar=True)

x_encoded_val = np.hstack((x_emb_val, x_sentimiento_val, x_url_val))
y_pred_val = random_search_xgb.predict(x_encoded_val)

accuracy_score(y_validation, y_pred_val)

In [ ]:
f1_score(y_validation, y_pred_val)

In [ ]:
x_sentimiento_test = mean_encoder.transform(test[['sentimiento']])
x_url_test = one_hot_encoder.transform(test[['tiene_url']]).toarray()

clean_text_test = test.text.map(limpiar_texto)
x_emb_test = sbert.encode(clean_text_test.tolist(), show_progress_bar=True)

x_encoded_test = np.hstack((x_emb_test, x_sentimiento_test, x_url_test))
y_pred_test = random_search_xgb.predict(x_encoded_test)

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'target': y_pred_test
})
submission.to_csv('./submission/xgb_sbert_submission.csv', index=False)

In [ ]:
fpr, tpr, _ = roc_curve(y_validation, y_pred_val)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_validation, y_pred_val):.2f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay


ConfusionMatrixDisplay(confusion_matrix(y_validation, y_pred_val)).plot(cmap='Blues')
plt.show()